In [0]:
%sql
CREATE TABLE IF NOT EXISTS capgeminipro.retail_gold.dim_customers (
    CustomerSK BIGINT GENERATED ALWAYS AS IDENTITY,
    CustomerID INT NOT NULL,
    CustomerName STRING,
    Email STRING,
    City STRING,
    Address STRING,
    StartDate DATE NOT NULL,
    EndDate DATE,
    IsCurrent STRING NOT NULL,
    CONSTRAINT pk_dim_customers PRIMARY KEY (CustomerSK)
) USING DELTA;

In [0]:
%sql
CREATE TABLE IF NOT EXISTS capgeminipro.retail_gold.dim_products (
    ProductSK BIGINT GENERATED ALWAYS AS IDENTITY,
    ProductID INT NOT NULL,
    ProductName STRING,
    Category STRING,
    UnitPrice DECIMAL(10,2),
    StartDate DATE NOT NULL,
    EndDate DATE,
    IsCurrent STRING NOT NULL,
    CONSTRAINT pk_dim_products PRIMARY KEY (ProductSK)
) USING DELTA;

In [0]:
%sql
CREATE TABLE IF NOT EXISTS capgeminipro.retail_gold.dim_stores (
    StoreSK BIGINT GENERATED ALWAYS AS IDENTITY,
    StoreID INT NOT NULL,
    StoreName STRING,
    Region STRING,
    StartDate DATE NOT NULL,
    EndDate DATE,
    IsCurrent STRING NOT NULL,
    CONSTRAINT pk_dim_stores PRIMARY KEY (StoreSK)
) USING DELTA;

In [0]:
%sql
CREATE TABLE IF NOT EXISTS capgeminipro.retail_gold.fact_sales (
    SalesSK BIGINT GENERATED ALWAYS AS IDENTITY,
    TransactionID INT NOT NULL,
    CustomerSK BIGINT NOT NULL,
    ProductSK BIGINT NOT NULL,
    StoreSK BIGINT NOT NULL,
    Quantity INT,
    TxnDate DATE,
    CONSTRAINT pk_fact_sales PRIMARY KEY (SalesSK)
) USING DELTA;

In [0]:
%sql
-- Step 1: Mark changed records as inactive
MERGE INTO capgeminipro.retail_gold.dim_customers tgt
USING (
    SELECT
        CustomerID,
        CustomerName,
        Email,
        City,
        Address,
        LastUpdated
    FROM capgeminipro.retail_silver.silver_customers
) src
ON tgt.CustomerID = src.CustomerID
AND tgt.IsCurrent = 'Y'

WHEN MATCHED AND (
       tgt.City <> src.City
    OR tgt.Address <> src.Address
    OR tgt.CustomerName <> src.CustomerName
    OR tgt.Email <> src.Email
)
THEN UPDATE SET
    tgt.EndDate = current_date() - 1,
    tgt.IsCurrent = 'N';

In [0]:
%sql
-- Step 2: Insert new versions of changed records AND new customers
INSERT INTO capgeminipro.retail_gold.dim_customers (
    CustomerID,
    CustomerName,
    Email,
    City,
    Address,
    StartDate,
    EndDate,
    IsCurrent
)
SELECT
    src.CustomerID,
    src.CustomerName,
    src.Email,
    src.City,
    src.Address,
    CURRENT_DATE() AS StartDate,
    NULL AS EndDate,
    'Y' AS IsCurrent
FROM capgeminipro.retail_silver.silver_customers src
LEFT JOIN capgeminipro.retail_gold.dim_customers tgt
    ON src.CustomerID = tgt.CustomerID
    AND tgt.IsCurrent = 'Y'
WHERE 
    -- New customers (not in target)
    tgt.CustomerID IS NULL
    OR
    -- Changed customers (attributes differ)
    (
        tgt.City <> src.City
        OR tgt.Address <> src.Address
        OR tgt.CustomerName <> src.CustomerName
        OR tgt.Email <> src.Email
    );

In [0]:
%sql
-- Step 1: Mark changed records as inactive
MERGE INTO capgeminipro.retail_gold.dim_products tgt
USING (
    SELECT
        ProductID,
        ProductName,
        Category,
        UnitPrice
    FROM capgeminipro.retail_silver.silver_products
) src
ON tgt.ProductID = src.ProductID
AND tgt.IsCurrent = 'Y'

WHEN MATCHED AND (
       tgt.ProductName <> src.ProductName
    OR tgt.Category <> src.Category
    OR tgt.UnitPrice <> src.UnitPrice
)
THEN UPDATE SET
    tgt.EndDate = current_date() - 1,
    tgt.IsCurrent = 'N';

In [0]:
%sql
-- Step 2: Insert new versions of changed records AND new products
INSERT INTO capgeminipro.retail_gold.dim_products (
    ProductID,
    ProductName,
    Category,
    UnitPrice,
    StartDate,
    EndDate,
    IsCurrent
)
SELECT
    src.ProductID,
    src.ProductName,
    src.Category,
    src.UnitPrice,
    CURRENT_DATE() AS StartDate,
    NULL AS EndDate,
    'Y' AS IsCurrent
FROM capgeminipro.retail_silver.silver_products src
LEFT JOIN capgeminipro.retail_gold.dim_products tgt
    ON src.ProductID = tgt.ProductID
    AND tgt.IsCurrent = 'Y'
WHERE 
    -- New products (not in target)
    tgt.ProductID IS NULL
    OR
    -- Changed products (attributes differ)
    (
        tgt.ProductName <> src.ProductName
        OR tgt.Category <> src.Category
        OR tgt.UnitPrice <> src.UnitPrice
    );

## 3. Stores Dimension (SCD2)

Tracks changes in store attributes: Name, Region

In [0]:
%sql
-- Step 1: Mark changed records as inactive
MERGE INTO capgeminipro.retail_gold.dim_stores tgt
USING (
    SELECT
        StoreID,
        StoreName,
        Region
    FROM capgeminipro.retail_silver.silver_stores
) src
ON tgt.StoreID = src.StoreID
AND tgt.IsCurrent = 'Y'

WHEN MATCHED AND (
       tgt.StoreName <> src.StoreName
    OR tgt.Region <> src.Region
)
THEN UPDATE SET
    tgt.EndDate = current_date() - 1,
    tgt.IsCurrent = 'N';

In [0]:
%sql
-- Step 2: Insert new versions of changed records AND new stores
INSERT INTO capgeminipro.retail_gold.dim_stores (
    StoreID,
    StoreName,
    Region,
    StartDate,
    EndDate,
    IsCurrent
)
SELECT
    src.StoreID,
    src.StoreName,
    src.Region,
    CURRENT_DATE() AS StartDate,
    NULL AS EndDate,
    'Y' AS IsCurrent
FROM capgeminipro.retail_silver.silver_stores src
LEFT JOIN capgeminipro.retail_gold.dim_stores tgt
    ON src.StoreID = tgt.StoreID
    AND tgt.IsCurrent = 'Y'
WHERE 
    -- New stores (not in target)
    tgt.StoreID IS NULL
    OR
    -- Changed stores (attributes differ)
    (
        tgt.StoreName <> src.StoreName
        OR tgt.Region <> src.Region
    );

## 4. Sales Fact Table (Incremental Load)

No SCD2 needed - just append new transactions

In [0]:
%sql
-- Insert only new transactions (not already in gold layer)
-- Use surrogate keys from dimension tables
INSERT INTO capgeminipro.retail_gold.fact_sales (
    TransactionID,
    CustomerSK,
    ProductSK,
    StoreSK,
    Quantity,
    TxnDate
)
SELECT
    src.TransactionID,
    dc.CustomerSK,
    dp.ProductSK,
    ds.StoreSK,
    src.Quantity,
    src.TxnDate
FROM capgeminipro.retail_silver.silver_sales src
INNER JOIN capgeminipro.retail_gold.dim_customers dc
    ON src.CustomerID = dc.CustomerID
    AND dc.IsCurrent = 'Y'
INNER JOIN capgeminipro.retail_gold.dim_products dp
    ON src.ProductID = dp.ProductID
    AND dp.IsCurrent = 'Y'
INNER JOIN capgeminipro.retail_gold.dim_stores ds
    ON src.StoreID = ds.StoreID
    AND ds.IsCurrent = 'Y'
LEFT JOIN capgeminipro.retail_gold.fact_sales tgt
    ON src.TransactionID = tgt.TransactionID
WHERE tgt.TransactionID IS NULL;

## 5. Verification

Check row counts and validate SCD2 logic

In [0]:
%sql
SELECT 
    'Customers' AS Dimension,
    COUNT(*) AS Total_Records,
    SUM(CASE WHEN IsCurrent = 'Y' THEN 1 ELSE 0 END) AS Active_Records,
    SUM(CASE WHEN IsCurrent = 'N' THEN 1 ELSE 0 END) AS Historical_Records
FROM capgeminipro.retail_gold.dim_customers

UNION ALL

SELECT 
    'Products' AS Dimension,
    COUNT(*) AS Total_Records,
    SUM(CASE WHEN IsCurrent = 'Y' THEN 1 ELSE 0 END) AS Active_Records,
    SUM(CASE WHEN IsCurrent = 'N' THEN 1 ELSE 0 END) AS Historical_Records
FROM capgeminipro.retail_gold.dim_products

UNION ALL

SELECT 
    'Stores' AS Dimension,
    COUNT(*) AS Total_Records,
    SUM(CASE WHEN IsCurrent = 'Y' THEN 1 ELSE 0 END) AS Active_Records,
    SUM(CASE WHEN IsCurrent = 'N' THEN 1 ELSE 0 END) AS Historical_Records
FROM capgeminipro.retail_gold.dim_stores;

In [0]:
%sql
SELECT 
    COUNT(*) AS Total_Transactions
FROM capgeminipro.retail_gold.fact_sales;

In [0]:
%sql
-- Show sample records with history (customers who have changed)
SELECT 
    CustomerID,
    CustomerName,
    City,
    Address,
    StartDate,
    EndDate,
    IsCurrent
FROM capgeminipro.retail_gold.dim_customers
WHERE CustomerID IN (
    SELECT CustomerID
    FROM capgeminipro.retail_gold.dim_customers
    GROUP BY CustomerID
    HAVING COUNT(*) > 1
)
ORDER BY CustomerID, StartDate
LIMIT 20;